# T6 — Demographic Analysis

Tests whether **gender**, **age**, or **shoe size** predict participants' cadence response to the tempo manipulation.

**Reads from existing exports** — run `01_pressure_analysis.ipynb` first to generate:
- `exports/initial_tempo_analysis/initial_tempo_stats.csv`
- `exports/figure5/influence_table.csv`

**Outcome variables:**

| Variable | Description |
|---|---|
| `pct_change` | % cadence change from baseline to trial end |
| `influence_score` | Trajectory-based coupling strength (0–1) |
| `correct_direction` | Whether cadence moved toward target (binary) |

**Configurations tested:** all sessions, split by direction (speeding up / slowing down) and condition order (tempo first / weight first)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

DATA_DIR   = Path('.')
EXPORT_DIR = DATA_DIR / 'exports'
FIG_DIR    = EXPORT_DIR / 'demographic_analysis'
FIG_DIR.mkdir(parents=True, exist_ok=True)

GENDER_COLORS = {'Female': '#e74c3c', 'Male': '#3498db'}
DIR_COLORS    = {'speeding up': '#3498db', 'slowing down': '#e74c3c'}

print('Setup complete.')

In [ ]:
# ── Load exported results ──────────────────────────────────────────────────────
df_tempo = pd.read_csv(DATA_DIR / 'exports/initial_tempo_analysis/initial_tempo_stats.csv')
df_tempo['pid'] = df_tempo['pid'].astype(str).str.zfill(2)

df_inf = pd.read_csv(DATA_DIR / 'exports/figure5/influence_table.csv', encoding='latin-1')
df_inf['pid'] = df_inf['pid'].astype(str).str.zfill(2)

# ── Load demographics from summary spreadsheet ────────────────────────────────
df_demo = pd.read_excel(DATA_DIR / 'Test 6_summary.xlsx', header=2)
df_demo['ID'] = df_demo['ID'].astype(str).str.zfill(2)
df_demo = df_demo[['ID', 'Gender', 'Age', 'UK shoe']].dropna(subset=['ID'])
df_demo.columns = ['pid', 'gender', 'age', 'shoe_size']
df_demo['shoe_size'] = pd.to_numeric(df_demo['shoe_size'], errors='coerce')
df_demo['age']       = pd.to_numeric(df_demo['age'],       errors='coerce')

# ── Merge ─────────────────────────────────────────────────────────────────────
df = (df_tempo
      .merge(df_inf[['pid', 'influence_score', 'profile', 'tier']], on='pid', how='left')
      .merge(df_demo, on='pid', how='left')
      .dropna(subset=['gender', 'age', 'shoe_size']))

df['gender_num']        = (df['gender'] == 'Male').astype(int)
df['correct_direction'] = df['correct_direction'].astype(float)

OUTCOMES = [
    ('pct_change',      '% cadence change'),
    ('influence_score', 'Influence score'),
]
CONFIGS = [
    ('All',          df),
    ('Speeding up',  df[df['direction'] == 'speeding up']),
    ('Slowing down', df[df['direction'] == 'slowing down']),
    ('Tempo first',  df[df['order'] == 'tempo first']),
    ('Weight first', df[df['order'] == 'weight first']),
]

print(f'Merged dataset: {len(df)} sessions')
print(f"Gender: {df['gender'].value_counts().to_dict()}")
print(f"Age:    {df['age'].min():.0f}–{df['age'].max():.0f}  (median {df['age'].median():.0f})")
print(f"Shoe:   {df['shoe_size'].min()}–{df['shoe_size'].max()}  (median {df['shoe_size'].median()})")
display(df[['pid', 'gender', 'age', 'shoe_size', 'direction', 'order',
            'pct_change', 'influence_score', 'correct_direction']].sort_values('pid'))

## Gender

Mann-Whitney U test comparing female vs male participants on each outcome, across all configurations.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(2, len(CONFIGS), figsize=(3.5 * len(CONFIGS), 8), sharey=False)
fig.subplots_adjust(hspace=0.45, wspace=0.3)

for row, (outcome, ylabel) in enumerate(OUTCOMES):
    for col, (cfg_name, cfg_df) in enumerate(CONFIGS):
        ax = axes[row][col]
        F = cfg_df[cfg_df['gender'] == 'Female'][outcome].dropna()
        M = cfg_df[cfg_df['gender'] == 'Male'][outcome].dropna()

        bp = ax.boxplot([F.values, M.values], positions=[0, 1], widths=0.45,
                        patch_artist=True,
                        medianprops=dict(color='black', lw=2),
                        whiskerprops=dict(lw=1), capprops=dict(lw=1),
                        flierprops=dict(marker='o', ms=3, alpha=0.5))
        for patch, color in zip(bp['boxes'], [GENDER_COLORS['Female'], GENDER_COLORS['Male']]):
            patch.set_facecolor(color); patch.set_alpha(0.45)
        for xi, (vals, color) in enumerate([(F, GENDER_COLORS['Female']), (M, GENDER_COLORS['Male'])]):
            jit = rng.uniform(-0.12, 0.12, len(vals))
            ax.scatter(xi + jit, vals, color=color, s=18, alpha=0.7, zorder=3)
        if len(F) > 1 and len(M) > 1:
            _, p = stats.mannwhitneyu(F, M, alternative='two-sided')
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            y_top = max(F.max(), M.max())
            ax.plot([0, 1], [y_top * 1.05, y_top * 1.05], color='black', lw=0.8)
            ax.text(0.5, y_top * 1.08, f'{sig} p={p:.3f}', ha='center', va='bottom', fontsize=7)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([f'F (n={len(F)})', f'M (n={len(M)})'], fontsize=8)
        if col == 0: ax.set_ylabel(ylabel, fontsize=9)
        if row == 0: ax.set_title(cfg_name, fontsize=9)
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        ax.axhline(0, color='#bdc3c7', lw=0.7)

fig.suptitle('Gender × outcome across configurations', fontsize=12)
plt.show()
fig.savefig(FIG_DIR / 'gender_analysis.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# Summary table
rows = []
for outcome, ylabel in OUTCOMES:
    for cfg_name, cfg_df in CONFIGS:
        F = cfg_df[cfg_df['gender'] == 'Female'][outcome].dropna()
        M = cfg_df[cfg_df['gender'] == 'Male'][outcome].dropna()
        if len(F) > 1 and len(M) > 1:
            _, p = stats.mannwhitneyu(F, M, alternative='two-sided')
        else:
            p = float('nan')
        rows.append({'outcome': ylabel, 'config': cfg_name,
                     'F median': round(float(F.median()), 3) if len(F) > 0 else None,
                     'M median': round(float(M.median()), 3) if len(M) > 0 else None,
                     'p (MWU)': round(p, 4), 'sig': '*' if p < 0.05 else ''})
display(pd.DataFrame(rows).set_index(['outcome', 'config']))

## Age and Shoe Size

Spearman correlation between each continuous demographic variable and the outcome measures, across all configurations.

UK shoe size is used as a proxy for foot size — it may correlate with stride characteristics and natural cadence variability.

In [ ]:
CONTINUOUS_VARS = [
    ('age',       'Age',          {}),
    ('shoe_size', 'UK shoe size', {}),
]

all_corr_rows = []

for demo_var, demo_label, _ in CONTINUOUS_VARS:
    fig, axes = plt.subplots(2, len(CONFIGS), figsize=(3.5 * len(CONFIGS), 7), sharey=False)
    fig.subplots_adjust(hspace=0.45, wspace=0.3)

    for row, (outcome, ylabel) in enumerate(OUTCOMES):
        for col, (cfg_name, cfg_df) in enumerate(CONFIGS):
            ax = axes[row][col]
            sub = cfg_df[[demo_var, outcome, 'direction']].dropna()

            for direction, color in DIR_COLORS.items():
                s = sub[sub['direction'] == direction]
                if len(s) < 2: continue
                ax.scatter(s[demo_var], s[outcome], color=color, s=30, alpha=0.7,
                           label=direction, edgecolors='white', linewidths=0.4)

            if len(sub) > 2:
                rho, p = stats.spearmanr(sub[demo_var], sub[outcome])
                sig = '*' if p < 0.05 else ''
                ax.text(0.03, 0.96, f'ρ={rho:+.2f}{sig}\np={p:.3f}',
                        transform=ax.transAxes, fontsize=7.5, va='top',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
                all_corr_rows.append({'demographic': demo_label, 'outcome': ylabel,
                                      'config': cfg_name, 'n': len(sub),
                                      'Spearman ρ': round(rho, 3), 'p': round(p, 4),
                                      'sig': '*' if p < 0.05 else ''})

            ax.axhline(0, color='#bdc3c7', lw=0.7)
            ax.set_xlabel(demo_label, fontsize=8)
            if col == 0: ax.set_ylabel(ylabel, fontsize=9)
            if row == 0: ax.set_title(cfg_name, fontsize=9)
            if row == 0 and col == 0: ax.legend(fontsize=7, loc='upper right')
            ax.grid(linestyle='--', alpha=0.3)

    fig.suptitle(f'{demo_label} × outcome (Spearman ρ)', fontsize=12)
    plt.show()
    fig.savefig(FIG_DIR / f'{demo_var}_analysis.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

df_corr = pd.DataFrame(all_corr_rows)
sig_only = df_corr[df_corr['sig'] == '*']
if len(sig_only):
    print('\nSignificant results (p<0.05):')
    display(sig_only.set_index(['demographic', 'outcome', 'config']))
else:
    print('\nNo significant results found (p<0.05) for age or shoe size.')

## Summary overview

Correlation matrix of all demographic variables against all outcome variables, across all sessions.

In [ ]:
DEMO_VARS    = [('age', 'Age'), ('shoe_size', 'Shoe size'), ('gender_num', 'Gender (M=1)')]
OUTCOME_VARS = [('pct_change', '% cadence change'), ('influence_score', 'Influence score'),
                ('correct_direction', 'Correct direction (0/1)')]

rho_mat = np.zeros((len(DEMO_VARS), len(OUTCOME_VARS)))
p_mat   = np.ones_like(rho_mat)

for i, (dv, _) in enumerate(DEMO_VARS):
    for j, (ov, _) in enumerate(OUTCOME_VARS):
        sub = df[[dv, ov]].dropna()
        if len(sub) > 2:
            rho_mat[i, j], p_mat[i, j] = stats.spearmanr(sub[dv], sub[ov])

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(rho_mat, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_xticks(range(len(OUTCOME_VARS)))
ax.set_yticks(range(len(DEMO_VARS)))
ax.set_xticklabels([o for _, o in OUTCOME_VARS], fontsize=9)
ax.set_yticklabels([d for _, d in DEMO_VARS], fontsize=9)
for i in range(len(DEMO_VARS)):
    for j in range(len(OUTCOME_VARS)):
        sig = '*' if p_mat[i, j] < 0.05 else ''
        ax.text(j, i, f'{rho_mat[i, j]:+.2f}{sig}', ha='center', va='center', fontsize=9,
                color='white' if abs(rho_mat[i, j]) > 0.35 else 'black')
ax.set_title('Spearman ρ: demographics × outcomes  (* p<0.05)', fontsize=11)
plt.tight_layout()
plt.show()
fig.savefig(FIG_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)

# Save merged dataset and full correlation table
df.drop(columns=['gender_num'], errors='ignore').to_csv(FIG_DIR / 'merged_demo_results.csv', index=False)
df_corr.to_csv(FIG_DIR / 'spearman_correlations.csv', index=False)
print(f'Saved figures and tables to {FIG_DIR}/')